Refer to https://github.com/GreenleafLab/HDMA/blob/main/code/05-misc/04-ChromBPNet_use_cases.ipynb

In [ ]:
%load_ext autoreload
%autoreload 2

# deal with matplotlib plots
%matplotlib inline

# display all outputs in a cell
get_ipython().ast_node_interactivity = 'all'

from IPython.display import display, HTML

In [ ]:
# misc
import sys
import os
import numpy as np
import random
from tqdm import tqdm
import pandas as pd

# io
import h5py
from pyfaidx import Fasta

# ML
import bpnetlite
from bpnetlite import ChromBPNet
from bpnetlite import BPNet
from bpnetlite.bpnet import CountWrapper
from bpnetlite.attribute import deep_lift_shap

import torch

import pickle

import pyranges as pr

In [ ]:
# plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn; seaborn.set_style('white')
import matplotlib.ticker as ticker
import plotnine as pn
import logomaker
import tangermeme
from tangermeme.plot import plot_logo

from tangermeme.utils import random_one_hot, _validate_input
from tangermeme.utils import characters
from tangermeme.ersatz import substitute
from tangermeme.utils import one_hot_encode

print(np.__version__)
print(tangermeme.__version__)
print(bpnetlite.__version__)
print(torch.__version__)

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES']='0'
#os.environ['CUDA_VISIBLE_DEVICES']='MIG-40f43250-998e-586a-ac37-d6520e92590f'

torch.manual_seed(100)

# editable text in PDFs
# https://stackoverflow.com/a/54111532
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

sys.path.append("~/HDMA/code/03-chrombpnet")
import tangermeme_utils as tanutils
from tangermeme_utils.utils import plot_motif
from tangermeme_utils.eval import summarize_across_folds, mean_across_sequences
from tangermeme_utils.wrappers import ChromBPNetWrapper, reshape_folds

In [ ]:
torch.cuda.device_count()

In [ ]:
dev = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
dev

torch.cuda.is_available()

t1 = torch.randn(1,2).to(dev)
t1
t1.device

In [ ]:
INPUTLEN = 2114
OUTPUTLEN = 1000
SHIFT_REL_INPUT = np.int64((2114 - 1000) / 2)
SHIFT_REL_INPUT

NARROWPEAK_SCHEMA = ["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"]

LOGO_ALPHABET = 'ACGT'

# finemo colorscheme
LOGO_COLORS = {"A": '#109648', "C": '#255C99', "G": '#F7B32B', "T": '#D62839'}

# logomaker colorscheme
LOGO_COLORS2= {
        'G': [1, .65, 0],
        'T': [1, 0, 0],
        'C': [0, 0, 1],
        'A': [0, .5, 0]
    }

CHAR_IGNORE = ['QWERYUIOPSDFHJKLZXVBNM']

In [ ]:
genome_fa = os.path.join("~/software/ref_tree_shrew/TreeShrew_scATAC_ref/fasta/genome.fa")

In [ ]:
TS_genome = Fasta(genome_fa)

In [ ]:
def predict_sequence(chrom,start,end,genome,subtype,models,models_counts,min_pos=557,max_pos=1557,show_fig=True):

    mid=(end+start)//2

    input_start=mid - INPUTLEN//2
    input_end=mid + INPUTLEN//2
    pre_seq = str(genome[chrom][mid - INPUTLEN//2 : mid + INPUTLEN//2])
    assert len(pre_seq) == 2114
    output_start=mid - OUTPUTLEN//2
    output_end=mid + OUTPUTLEN//2
    

    #使用 .upper() 转换为大写
    seq_ohe = one_hot_encode(pre_seq.upper()).type(torch.float32).unsqueeze(0) #seq_ohe.shape，torch.Size([1, 4, 2114])。1，batch size,一次输入一条序列；4，四个碱基通道；2114，序列长度。
    # predict
    print("@ predicting...")
    seq_pred_list = []

    for model in models:
        y_seq = tangermeme.predict.predict(model, seq_ohe)
        seq_pred_list.append(y_seq)

    seq_pred = reshape_folds(seq_pred_list)

    seq_pred_profile_mean, _ = summarize_across_folds(seq_pred[0])

    seq_pred_profile_mean.shape

    # compute attribution
    print("@ computing attribution...")
    try:

        seq_attr_list = []
        
        for model in models_counts:
            y_seq = deep_lift_shap(model, seq_ohe, random_state=0)#torch.Size([1, 4, 2114])
            seq_attr_list.append(y_seq)

        seq_attr = torch.stack(seq_attr_list, dim=0)#torch.Size([5, 1, 4, 2114])。5，5个fold。
        seq_attr_mean, _ = summarize_across_folds(seq_attr)#5 个 fold 求平均。torch.Size([1, 4, 2114])。

        
    except Exception as e:
        print(f"Error during attribution computation: {e}")

    results={"seq_pred_profile_mean":seq_pred_profile_mean,
        "seq_attr_mean":seq_attr_mean}

    pred_seq = results['seq_pred_profile_mean'].squeeze()

    attr_seq = results['seq_attr_mean'].sum(axis = 1).squeeze().T[(INPUTLEN // 2 - 500):(INPUTLEN // 2 + 500)]

    #输出维度信息
    print("seq_ohe shape:", seq_ohe.shape) #torch.Size([1, 4, 2114])
    print("single-fold profile shape:", seq_pred_list[0][0].shape) #torch.Size([1, 1, 1000])
    print("single-fold attribution shape:", seq_attr_list[0].shape) #torch.Size([1, 4, 2114])
    print("all-fold profile shape:", seq_pred[0].shape) #torch.Size([5, 1, 1, 1000])
    print("all-fold attribution shape:", seq_attr.shape) #torch.Size([5, 1, 4, 2114])
    print("mean profile shape:", seq_pred_profile_mean.shape) #torch.Size([1, 1, 1000])
    print("mean attribution shape:", seq_attr_mean.shape) #torch.Size([1, 4, 2114])
    print("pred_seq shape:", pred_seq.shape) #torch.Size([1000])
    print("attr_seq shape:", attr_seq.shape) #torch.Size([1000])

    # ploting
    fig, ax = plt.subplots(3, 1, figsize = (25, 6))
    alpha = 0.3
    # prediction in 1,000 bp
    ax[0].plot(pred_seq, linewidth = 0.5, label = "seq(5-fold mean)", color = "blue")
    ax[0].set_ylabel("Pred. \naccessibility")
    ax[0].axvspan(min_pos-557, max_pos-557, alpha = alpha, color='yellow')
    ax[0].legend(fontsize = 8)
    # attribution in 1,000 bp
    ax[1].plot(attr_seq, linewidth = 0.5, label = "seq(5-fold mean)", color = "blue")
    ax[1].set_ylabel("Contrib. \nscores")
    ax[1].axvspan(min_pos-557, max_pos-557, alpha = alpha, color = 'yellow')
    ax[1].legend(fontsize = 8)
    #
    ylim = None
    tangermeme.plot.plot_logo(np.float64(results['seq_attr_mean'][0]), start = min_pos, end = max_pos, ax=ax[2], ylim = ylim) #results['seq_attr_mean'][0].shape,torch.Size([4, 2114])，画logo是传入四个碱基通道的贡献值 #end 是开区间，不包含 end 本身。
    ax[2].set_ylabel("Contrib. \n")
        # put a title at the very top
    #plt.suptitle(subtype + ":" + chrom + ":"+ str(start+1) + "-" + str(end), fontsize = 12)
    plt.suptitle(subtype + ":" + chrom + ":"+ str(output_start+1) + "-" + str(output_end), fontsize = 12)
    plt.xlabel("Genomic position (bp)")
    plt.tight_layout()

    # ============================================================
    # 计算与 plot_logo 完全一致区间内的 contribution score，得到最大值和最小值
    # ============================================================

    contribution_region=results['seq_attr_mean'].sum(axis = 1).squeeze().T[min_pos:max_pos] #python切片规则，左闭右开 [start, end)
    # 最大值和最小值
    contribution_max = contribution_region.max().item()
    contribution_min = contribution_region.min().item()

    print("Maximum contribution score:", contribution_max)
    print("Minimum contribution score:", contribution_min)
    
    if show_fig:
        plt.show()
    else:
        #plt.savefig(os.path.join(outdir, subtype + ":" + chrom + ":"+ str(start+1) + "-" + str(end) + ".pdf"))
        plt.savefig(os.path.join(outdir, subtype + ":" + chrom + ":"+ str(output_start+1) + "-" + str(output_end) + ".pdf"))

    plt.close(fig)
    

In [ ]:
outdir="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_marker_region_predict"

In [ ]:
proj_in="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C7Endothelial_cells/output/models"

In [ ]:
#load model
models = [
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C7.1/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C7.1/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C7.1/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C7.1/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C7.1/models/chrombpnet_nobias.h5")))
]

models_counts = [
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C7.1/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C7.1/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C7.1/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C7.1/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C7.1/models/chrombpnet_nobias.h5")))
]

chr15:58,115,498-58,116,489	chr15:58,177,297-58,178,044

In [ ]:
#1.chr15:58,115,498-58,116,489
chrom="chr15"
start=58115498
end=58116489
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C7.1_region1",models=models,models_counts=models_counts,min_pos=250+150+557,max_pos=750-100+557,show_fig=True)

In [ ]:
#2.chr15:58,177,297-58,178,044
chrom="chr15"
start=58177297
end=58178044
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C7.1_region2",models=models,models_counts=models_counts,min_pos=250+100+557,max_pos=750-150+557,show_fig=True)

In [ ]:
proj_in="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C16Neural_crest_PNS_glia/output/models"

In [ ]:
#load model
models = [
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C16.5/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C16.5/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C16.5/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C16.5/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C16.5/models/chrombpnet_nobias.h5")))
]

models_counts = [
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C16.5/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C16.5/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C16.5/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C16.5/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C16.5/models/chrombpnet_nobias.h5")))
]

chr6:110,003,896-110,004,484	chr6:109,933,216-109,934,063	chr6:109,944,937-109,945,689	chr6:109,954,091-109,954,893

In [ ]:
#1.chr6:110,003,896-110,004,484
chrom="chr6"
start=110003896
end=110004484
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C16.5_region1",models=models,models_counts=models_counts,min_pos=250+100+557,max_pos=750-150+557,show_fig=True)

In [ ]:
#2.chr6:109,933,216-109,934,063
chrom="chr6"
start=109933216
end=109934063
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C16.5_region2",models=models,models_counts=models_counts,min_pos=250+150+557,max_pos=750-100+557,show_fig=True)

In [ ]:
#3.chr6:109,944,937-109,945,689
chrom="chr6"
start=109944937
end=109945689
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C16.5_region3",models=models,models_counts=models_counts,min_pos=250+125+557,max_pos=750-125+557,show_fig=True)

In [ ]:
#4.chr6:109,954,091-109,954,893
chrom="chr6"
start=109954091
end=109954893
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C16.5_region4",models=models,models_counts=models_counts,min_pos=250+25+557,max_pos=750-225+557,show_fig=True)

In [ ]:
proj_in="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C6Definitive_erythroid_cells/output/models"

In [ ]:
#load model
models = [
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C6.1/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C6.1/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C6.1/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C6.1/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C6.1/models/chrombpnet_nobias.h5")))
]

models_counts = [
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C6.1/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C6.1/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C6.1/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C6.1/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C6.1/models/chrombpnet_nobias.h5")))
]

chr11:64,391,757-64,392,353

In [ ]:
#1.chr11:64,391,757-64,392,353
chrom="chr11"
start=64391757
end=64392353
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C6.1_region1",models=models,models_counts=models_counts,min_pos=250+125+557,max_pos=750-125+557,show_fig=True)

In [ ]:
proj_in="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C10Hepatocytes/output/models"

In [ ]:
#load model
models = [
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C10.1/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C10.1/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C10.1/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C10.1/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C10.1/models/chrombpnet_nobias.h5")))
]

models_counts = [
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C10.1/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C10.1/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C10.1/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C10.1/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C10.1/models/chrombpnet_nobias.h5")))
]

chr9:63,712,177-63,712,638	chr9:63,697,351-63,697,818

In [ ]:
#1.chr9:63,712,177-63,712,638
chrom="chr9"
start=63712177
end=63712638
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C10.1_region1",models=models,models_counts=models_counts,min_pos=250+125+557,max_pos=750-125+557,show_fig=True)

In [ ]:
#2.chr9:63,697,351-63,697,818
chrom="chr9"
start=63697351
end=63697818
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C10.1_region2",models=models,models_counts=models_counts,min_pos=250+150+557,max_pos=750-100+557,show_fig=True)

In [ ]:
proj_in="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C23Retinal_cells/output/models"

In [ ]:
#load model
models = [
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C23.6/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C23.6/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C23.6/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C23.6/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C23.6/models/chrombpnet_nobias.h5")))
]

models_counts = [
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C23.6/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C23.6/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C23.6/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C23.6/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C23.6/models/chrombpnet_nobias.h5")))
]

chr12:18,493,526-18,493,921	
chr12:18,497,788-18,498,220	
chr12:18,485,733-18,486,321	
chr12:18,465,331-18,467,823	
chr12:18,479,096-18,479,617

In [ ]:
#1.chr12:18,493,526-18,493,921
chrom="chr12"
start=18493526
end=18493921
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C23.6_region1",models=models,models_counts=models_counts,min_pos=250+125+557,max_pos=750-125+557,show_fig=True)

In [ ]:
#2.chr12:18,497,788-18,498,220
chrom="chr12"
start=18497788
end=18498220
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C23.6_region2",models=models,models_counts=models_counts,min_pos=250+100+557,max_pos=750-150+557,show_fig=True)

In [ ]:
#3.chr12:18,485,733-18,486,321
chrom="chr12"
start=18485733
end=18486321
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C23.6_region3",models=models,models_counts=models_counts,min_pos=250+150+557,max_pos=750-100+557,show_fig=True)

In [ ]:
#4.chr12:18,465,331-18,467,823
chrom="chr12"
start=18465331
end=18467823
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C23.6_region4",models=models,models_counts=models_counts,min_pos=200+557,max_pos=450+557,show_fig=True)

In [ ]:
#5.chr12:18,479,096-18,479,617
chrom="chr12"
start=18479096
end=18479617
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,TS_genome,subtype="C23.6_region5",models=models,models_counts=models_counts,min_pos=250+50+557,max_pos=750-200+557,show_fig=True)